# 💳 Credit Card Fraud Detection
**Concepts:** Classification | Imbalanced Data (SMOTE) | Feature Scaling | Confusion Matrix

In [ ]:
# ─── 1. Install dependencies ───────────────────────────────────────────────
!pip install imbalanced-learn --quiet

In [ ]:
# ─── 2. Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')
print('✅ All imports successful!')

In [ ]:
# ─── 3. Generate a realistic imbalanced dataset ────────────────────────────
# Simulates a credit-card dataset: 98% legit, 2% fraud
X, y = make_classification(
    n_samples=10_000,
    n_features=20,
    n_informative=10,
    weights=[0.98, 0.02],   # <-- heavy class imbalance
    flip_y=0,
    random_state=42
)

feature_names = [f'V{i}' for i in range(1, 21)]
df = pd.DataFrame(X, columns=feature_names)
df['Class'] = y  # 0 = Legit, 1 = Fraud

print('Dataset shape:', df.shape)
print('\nClass distribution:')
print(df['Class'].value_counts().rename({0: 'Legit', 1: 'Fraud'}))

In [ ]:
# ─── 4. Visualise class imbalance ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3))
counts = df['Class'].value_counts()
ax.bar(['Legit (0)', 'Fraud (1)'], counts.values,
       color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title('Class Distribution (Before SMOTE)', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 5. Feature Scaling ────────────────────────────────────────────────────
X_raw = df.drop('Class', axis=1).values
y_raw = df['Class'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print('Feature scaling done — mean ≈ 0, std ≈ 1')
print(f'  Sample mean after scaling: {X_scaled.mean():.4f}')
print(f'  Sample std  after scaling: {X_scaled.std():.4f}')

In [ ]:
# ─── 6. Train / Test Split ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)
print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

In [ ]:
# ─── 7. Handle Imbalance with SMOTE (Experiment 9 concept) ────────────────
print('Before SMOTE:', np.bincount(y_train))

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)

print('After  SMOTE:', np.bincount(y_res))
print('✅ Classes are now balanced!')

In [ ]:
# ─── 8. Train Models ───────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

trained = {}
for name, model in models.items():
    model.fit(X_res, y_res)
    trained[name] = model
    print(f'✅ {name} trained')

In [ ]:
# ─── 9. Confusion Matrix & Classification Report ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, model) in zip(axes, trained.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    print(f'\n── {name} ──')
    print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 10. ROC-AUC Curve (Bonus) ─────────────────────────────────────────────
plt.figure(figsize=(7, 5))

colors = ['darkorange', 'steelblue']
for (name, model), color in zip(trained.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Fraud Detection', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 11. Feature Importance (Random Forest) ────────────────────────────────
rf = trained['Random Forest']
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 4))
importances.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Importance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## ✅ Summary
| Step | What we did |
|---|---|
| **Dataset** | Simulated 10 000 transactions, 2% fraud |
| **Feature Scaling** | `StandardScaler` — zero mean, unit variance |
| **Imbalance handling** | `SMOTE` to oversample minority (fraud) class |
| **Models** | Logistic Regression & Random Forest |
| **Evaluation** | Confusion Matrix, Classification Report, ROC-AUC |